In [36]:
import os
import sys
from dotenv import load_dotenv
from sklearn.metrics import adjusted_mutual_info_score
load_dotenv() 

# Set the target folder name you want to reach
target_folder = "NCEAS_Unsupervised_NLP"

# Get the current working directory
current_dir = os.getcwd()

# Loop to move up the directory tree until we reach the target folder
while os.path.basename(current_dir) != target_folder:
    parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
    if parent_dir == current_dir:
        # If we reach the root directory and haven't found the target, exit
        raise FileNotFoundError(f"{target_folder} not found in the directory tree.")
    current_dir = parent_dir

# Change working directory to the project root
os.chdir(current_dir)

# Add the "NCEAS_Unsupervised_NLPt" directory to sys.path
sys.path.insert(0, current_dir)

In [37]:
import pandas as pd

In [38]:
data_sources = {
    "Qwen3-Embedding-0.6B": "Qwen3-Embedding-0.6B_results/qwen3_arxiv_results.csv",
    "all-MiniLM-L6-v2": "all-MiniLM-L6-v2_results/all_MiniLM_other_arxiv_results.csv"
}

In [39]:
results = []
maximum = False

for model, filepath in data_sources.items():

    df = pd.read_csv(filepath)
    df = df.fillna("None")

    if 'reduction_params' in df.columns:
        df['Params'] = df['reduction_params'] + df['cluster_params']
    elif 'Params' not in df.columns:
        df['Params'] = ['None'] * len(df)

    if maximum:

        grouped_mean = df.groupby(
            ['reduction_method','cluster_method','level','Params']
        )[['ARI']].mean().reset_index()

        grouped_max = grouped_mean.groupby(
            ['reduction_method','cluster_method']
        )[['ARI']].max().reset_index()

        grouped_max['embedding_model'] = model
        results.append(grouped_max)

    else:

        grouped_median = df.groupby(
            ['reduction_method','cluster_method','level']
        )[['ARI']].median().reset_index()

        grouped_mean = grouped_median.groupby(
            ['reduction_method','cluster_method']
        )[['ARI']].mean().reset_index()

        grouped_mean['embedding_model'] = model
        results.append(grouped_mean)

final_df = pd.concat(results, ignore_index=True)

In [40]:
final_df = final_df.replace({"DC":"Diffusion condensation"})
final_df = final_df.replace({"Diffusion Condensation":"Diffusion condensation"})

final_df = final_df[final_df['reduction_method']!="BASE-PHATE"]
final_df = final_df[final_df['reduction_method']!="None"]
final_df = final_df[final_df['reduction_method']!="tSNE"]

final_df

,reduction_method,cluster_method,ARI,embedding_model
0,PCA,Agglomerative,0.478989,Qwen3-Embedding-0.6B
1,PCA,Diffusion condensation,0.082603,Qwen3-Embedding-0.6B
2,PCA,HDBSCAN,0.000018,Qwen3-Embedding-0.6B
3,PHATE,Agglomerative,0.476800,Qwen3-Embedding-0.6B
4,PHATE,Diffusion condensation,0.296899,Qwen3-Embedding-0.6B
5,PHATE,HDBSCAN,-0.000139,Qwen3-Embedding-0.6B
6,UMAP,Agglomerative,0.490207,Qwen3-Embedding-0.6B
7,UMAP,Diffusion condensation,0.112346,Qwen3-Embedding-0.6B
8,UMAP,HDBSCAN,0.428488,Qwen3-Embedding-0.6B
9,PCA,Agglomerative,0.263159,all-MiniLM-L6-v2


In [41]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
final_df[final_df['cluster_method']=="Diffusion Condensation"].sort_values(by='ARI',ascending= False)


,reduction_method,cluster_method,ARI,embedding_model


In [42]:
import matplotlib.pyplot as plt

df = final_df

reduction_order = ['PHATE', 'PCA', 'UMAP', 'T-SNE']
cluster_order = ['Diffusion condensation', 'Agglomerative','HDBSCAN']
embedding_order = ['Qwen3-Embedding-0.6B','all-MiniLM-L6-v2']

df['reduction_method'] = pd.Categorical(df['reduction_method'], categories=reduction_order, ordered=True)
df['cluster_method'] = pd.Categorical(df['cluster_method'], categories=cluster_order, ordered=True)
df['embedding_model'] = pd.Categorical(df['embedding_model'], categories=embedding_order, ordered=True)

pivot = df.pivot_table(
    index=['reduction_method','cluster_method'],
    columns='embedding_model',
    values='ARI'
)

pivot = pivot.sort_index()

/var/folders/1s/gzgswgnn7td1g02_q1hrvy3w0000gn/T/ipykernel_27621/2940146382.py:13: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(


In [43]:
pivot

embedding_model                          Qwen3-Embedding-0.6B  \
reduction_method cluster_method                                 
PHATE            Diffusion condensation              0.296899   
                 Agglomerative                       0.476800   
                 HDBSCAN                            -0.000139   
PCA              Diffusion condensation              0.082603   
                 Agglomerative                       0.478989   
                 HDBSCAN                             0.000018   
UMAP             Diffusion condensation              0.112346   
                 Agglomerative                       0.490207   
                 HDBSCAN                             0.428488   

embedding_model                          all-MiniLM-L6-v2  
reduction_method cluster_method                            
PHATE            Diffusion condensation          0.080199  
                 Agglomerative                   0.503058  
                 HDBSCAN                         0.028377  
PCA              Diffusion condensation          0.077261  
                 Agglomerative                   0.263159  
                 HDBSCAN                         0.000164  
UMAP             Diffusion condensation          0.091755  
                 Agglomerative                   0.498970  
                 HDBSCAN                        -0.000349